# Instrument Primer — NIRISS GR150R / F150W

This notebook explains how the NIRISS GR150R grism disperses light onto the detector. It covers the active diffraction orders, their sensitivity, trace geometry, dispersion law, and which source positions contribute to a given image stamp. No forward-operator build is required — all computations use `config.get_trace()` directly.

> **Dependency note:** Trace geometry is provided by the [`grismagic`](https://github.com/your-org/grismagic) package via `spectrex.InstrumentConfig`. The dispersion coefficients come from the `GR150R.F150W.220725.conf` calibration file.

## §0 — Setup

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import spectrex
from spectrex import EigenspectraBasis, InstrumentConfig
from spectrex.instrument import _ORDER_LETTER_TO_INT

# ── Paths ────────────────────────────────────────────────────────────────────
TESTDATA = Path('../testdata')
IMAGE_SHAPE = (500, 20)   # detector stamp used throughout
SOURCE_I, SOURCE_J = 250, 10  # reference source position

# ── Colour palette (consistent across all figures) ───────────────────────────
ORDER_COLOURS = {'A': '#4da6ff', 'B': '#ff9f40', 'C': '#c08cff'}
ORDER_LABELS  = {'A': 'Order A (1st, dispersed)',
                 'B': 'Order B (0th, undispersed)',
                 'C': 'Order C (2nd, dispersed)'}

# ── Load instrument config ────────────────────────────────────────────────────
config = InstrumentConfig.from_files(
    conf_path=TESTDATA / 'Config Files' / 'GR150R.F150W.220725.conf',
    wavelengthrange_path=TESTDATA / 'jwst_niriss_wavelengthrange_0002.asdf',
    sensitivity_dir=TESTDATA / 'SenseConfig' / 'wfss-grism-configuration',
    filter_name='F150W',
    n_wavelengths=150,
)

basis = EigenspectraBasis.from_csv(
    TESTDATA / 'eigenspectra_kurucz.csv',
    config.wavelengths,
)

# Active orders: those with calibrated sensitivity
active_orders = [o for o in config.orders if o in config.sensitivity]

# ── Summary table ────────────────────────────────────────────────────────────
print(f'spectrex {spectrex.__version__}')
print(f'Grism : {config.grism}')
print(f'Filter: {config.filter_name}')
print(f'λ range: {config.wavelengths[0]:.0f} – {config.wavelengths[-1]:.0f} Å  '
      f'({len(config.wavelengths)} samples)')
print()
print(f'{"Order":<8}{"Integer":<10}{"Physical meaning":<28}{"Has sensitivity"}')
print('-' * 60)
meaning = {0: '0th (undispersed)', 1: '1st (dispersed)', 2: '2nd (dispersed)'}
for o in config.orders:
    idx = _ORDER_LETTER_TO_INT.get(o, '?')
    has_sens = 'yes' if o in config.sensitivity else 'no'
    print(f'{o:<8}{str(idx):<10}{meaning.get(idx, "unknown"):<28}{has_sens}')


spectrex 0.2.1.dev18+gc6f5edaa6.d20260506
Grism : GR150R
Filter: F150W
λ range: 12900 – 17100 Å  (150 samples)

Order   Integer   Physical meaning            Has sensitivity
------------------------------------------------------------
A       1         1st (dispersed)             yes
B       0         0th (undispersed)           yes
C       2         2nd (dispersed)             yes
D       ?         unknown                     no
E       ?         unknown                     no
